# conv-output-shape — worked example 3: Trace spatial shape through a stack of conv layers

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `conv-output-shape`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

When conv layers are stacked, the output spatial size of one layer becomes the input size of the next. Apply `L_out = (L + 2P - K)//S + 1` repeatedly, threading each layer's result forward. Strided layers shrink the map by roughly the stride factor each time, so spatial size decays multiplicatively through the stack.

## Worked solution

We trace a square feature map of side `64` through three conv layers, all square so we track one spatial dimension.

Layer specs `(K, S, P)`: L1 = (3, 1, 1), L2 = (3, 2, 1), L3 = (5, 2, 0).

**Step 1 — Layer 1.** `(64 + 2*1 - 3)//1 + 1 = 63//1 + 1 = 63 + 1 = 64`. Kernel 3, stride 1, padding 1 is SAME, so the size stays 64.

**Step 2 — Layer 2.** Feed 64 in: `(64 + 2*1 - 3)//2 + 1 = 63//2 + 1 = 31 + 1 = 32`. Stride 2 halves the map (the floor handles the odd 63). Now 32.

**Step 3 — Layer 3.** Feed 32 in: `(32 + 2*0 - 5)//2 + 1 = 27//2 + 1 = 13 + 1 = 14`. No padding plus a larger kernel trims extra border, giving 14.

**Step 4 — final.** The 64x64 map became 14x14 after the stack: `64 -> 64 -> 32 -> 14`.

**Why it works.** Each layer only sees the spatial size handed to it; there is no global formula for a stack, only repeated application. We verify by chaining three real `nn.Conv2d` layers and reading the final shape.

In [ ]:
def trace_conv_stack(L, layers):
    for (K, S, P) in layers:
        L = (L + 2 * P - K) // S + 1
    return L


layers = [(3, 1, 1), (3, 2, 1), (5, 2, 0)]
pred = trace_conv_stack(64, layers)

net = t.nn.Sequential(
    t.nn.Conv2d(3, 8, 3, stride=1, padding=1),
    t.nn.Conv2d(8, 16, 3, stride=2, padding=1),
    t.nn.Conv2d(16, 32, 5, stride=2, padding=0),
)
actual = net(t.zeros(1, 3, 64, 64)).shape[-1]
print("predicted side:", pred)
print("actual side:   ", actual)
print("match:", pred == actual)